# Lab 5.2 &mdash; Build a Tool: Give the Agent Hands

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Represent a tool as a dict with a name, a description and a function
- Build a registry that maps tool names to tools
- Implement a SAFE calculator (no bare eval) and a lookup tool

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; Tools turn an LLM into an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-02"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A model alone only emits **text**. A **tool** is a plain function you expose to it, described by a
**name**, a **description** (which the model reads to decide when to use it) and the **function**
itself. We collect tools in a **registry** keyed by name. Two staples: a **calculator** (LLMs are
bad at exact math &mdash; offload it) and a **lookup** over a small fixed knowledge base.

> **Safety:** never run bare `eval()` on free text. Our calculator uses a tiny **AST-based** safe
> evaluator that only allows numbers and arithmetic operators.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
print("safe_calc('2+2') =", safe_calc("2+2"), "| safe_calc('3*(4+1)') =", safe_calc("3*(4+1)"))
# A bare eval() would happily run safe_calc("__import__('os').system('rm -rf /')") -- ours refuses.

## Your Turn
Implement the two tool functions, then build the registry mapping each tool's name to its dict.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

KNOWLEDGE = {"capital of france": "Paris", "population of metropolis": "120000"}

def calculator_fn(expr):
    return ___    # TODO: evaluate expr with the safe calculator

def lookup_fn(key):
    return ___    # TODO: look key up in KNOWLEDGE (lowercased/stripped), default "unknown"

calculator = {"name": "calculator", "description": "evaluate arithmetic like 2+2", "fn": calculator_fn}
lookup     = {"name": "lookup", "description": "look up a known fact by key", "fn": lookup_fn}

def build_registry(tools):
    return ___    # TODO: a dict mapping each tool's name to the tool

REGISTRY = {}
try:
    REGISTRY = build_registry([calculator, lookup])
    print('tools:', list(REGISTRY))
    print('calculator(2+2) =', REGISTRY['calculator']['fn']('2+2'))
    print('lookup(capital of france) =', REGISTRY['lookup']['fn']('capital of france'))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("registry holds both tools", lambda: set(REGISTRY) >= {"calculator", "lookup"})
expect_true("each tool has name/description/fn", lambda: all({"name","description","fn"} <= set(t) for t in REGISTRY.values()))
expect_true("calculator tool computes 2+2 == 4", lambda: REGISTRY["calculator"]["fn"]("2+2") == 4)
expect_true("lookup tool finds a known key", lambda: REGISTRY["lookup"]["fn"]("capital of france") == "Paris")
expect_true("lookup returns 'unknown' for a missing key", lambda: REGISTRY["lookup"]["fn"]("price of gold") == "unknown")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A tool is just `{name, description, fn}` in a registry. That description is what a real LLM reads to choose a tool &mdash; next we make the agent route to the right one.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>